# SkillBridge: Exploratory Data Analysis & Graph Verification
This notebook demonstrates the end-to-end mathematical and NLP workflow of **SkillBridge**:
1. Skill Vocabulary and spaCy PhraseMatcher Extraction
2. Job Postings Ingestion and Role Normalization
3. Role-Based Co-Occurrence Graph Construction (NetworkX)
4. Top Neighbors Analysis for "React" and "Docker"
5. Resume Parsing, Dual-Fit Scoring, and Next-Best-Skill Recommendation

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import networkx as nx

# Ensure project root is in sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.skill_extract import SkillExtractor
from src.demand import compute_role_demand, get_role_demand_dict
from src.graph_builder import build_and_save_all_graphs, get_top_neighbors, load_graph
from src.gap import analyze_resume_gap
from src.recommend import recommend_next_skills
from src.readability import score_resume_readability

print("SkillBridge core modules successfully imported!")

## 1. Skill Vocabulary Inspection

In [ ]:
extractor = SkillExtractor()
meta_df = extractor.skill_meta_df
print(f"Total canonical skills in vocabulary: {len(meta_df)}")
print("\nDistribution by Skill Category:")
display(meta_df['category'].value_counts())

## 2. Test Skill Extraction with Real-world Text

In [ ]:
sample_text = "Experienced with python, pytorch, k8s, docker, postgres, and aws building restful microservices."
skills = extractor.extract_skills(sample_text)
print("Input Text:", sample_text)
print("Extracted Canonical Skills:", skills)

## 3. Co-Occurrence Graph Exploration & Neighbors

In [ ]:
# Load Frontend Developer Graph
fe_graph = load_graph("Frontend Developer")
print(f"Frontend Developer Graph: {fe_graph.number_of_nodes()} nodes, {fe_graph.number_of_edges()} edges")

print("\nTop co-occurring skills for 'React':")
for neighbor, wt in get_top_neighbors(fe_graph, "React", top_n=8):
    print(f"  - {neighbor:<16}: {wt} co-occurrences")

# Load DevOps Engineer Graph
devops_graph = load_graph("DevOps Engineer")
print(f"\nDevOps Engineer Graph: {devops_graph.number_of_nodes()} nodes, {devops_graph.number_of_edges()} edges")
print("\nTop co-occurring skills for 'Docker':")
for neighbor, wt in get_top_neighbors(devops_graph, "Docker", top_n=8):
    print(f"  - {neighbor:<16}: {wt} co-occurrences")

## 4. End-to-End Recommendation Test

In [ ]:
sample_student_resume = """
EDUCATION
B.Tech CSE state university

SKILLS
Python, Pandas, Scikit-Learn, Git, SQL

PROJECTS
- Built customer churn prediction model with Python and Scikit-Learn.
- Automated SQL queries for data preprocessing.
"""

ml_graph = load_graph("Machine Learning Engineer")
demand_df = pd.read_csv(ROOT_DIR / "data" / "processed" / "role_demand.csv")
demand_dict = get_role_demand_dict(demand_df, "Machine Learning Engineer")

gap_res = analyze_resume_gap(
    resume_text=sample_student_resume,
    target_role="Machine Learning Engineer",
    job_skills_df=pd.read_parquet(ROOT_DIR / "data" / "processed" / "job_skills.parquet"),
    demand_dict=demand_dict,
    extractor=extractor
)

recs = recommend_next_skills(
    user_skills=gap_res["user_skills"],
    missing_skills=gap_res["missing_skills"],
    G=ml_graph,
    demand_dict=demand_dict,
    extractor=extractor,
    top_k=3
)

print(f"Overall Match Score: {gap_res['overall_match_score']}%")
print(f"Set Coverage Score: {gap_res['set_coverage_score']}%")
print("\nTop 3 Recommended Next Skills:")
for r in recs:
    print(f"- {r['skill']} (Score: {r['score']}): {r['reason']}")